In [ ]:
from pyspark.sql import functions as F

df = spark.table("desafio_vendas.bronze.vendas_raw")
colunas_negocio = [c for c in df.columns if not c.startswith("_")]
total = df.count()
print(f"Total de linhas: {total}")

In [ ]:
resumo_nulos = df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in colunas_negocio
])
display(resumo_nulos)

In [ ]:
dist = df.select(colunas_negocio).distinct().count()
print(f"Linhas distintas (todas as colunas de negócio): {dist}")
print(f"Duplicatas exatas: {total - dist}")

# Mesma venda com chave natural repetida (cliente + produto + data da venda)
chave = ["cliente", "produto", "data_venda"]
display(
    df.groupBy(chave).count().filter("count > 1")
      .orderBy(F.desc("count")).limit(10)
)

In [ ]:
categoricas = ["produto", "tipo_produto", "status_envio", "estado", "cliente_sexo",
               "status_venda", "tipo_pagamento", "status_desconto", "item_promocional",
               "avaliacao_produto"]

for c in categoricas:
    print(f"\n=== {c} ===")
    display(df.groupBy(c).count().orderBy(F.desc("count")))

In [ ]:
def padrao_data(col):
    return (F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), "vazio")
             .when(F.col(col).rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
             .when(F.col(col).rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
             .when(F.col(col).rlike(r"^\d{2}-\d{2}-\d{4}$"), "xx-xx-yyyy")
             .otherwise("outro"))

for c in ["data_venda", "data_retirada"]:
    print(f"\n=== formatos de {c} ===")
    display(df.groupBy(padrao_data(c).alias("formato")).count())

# No formato xx-xx-yyyy: quantos são MM-dd, quantos são dd-MM e quantos são ambíguos?
x = df.filter(F.col("data_venda").rlike(r"^\d{2}-\d{2}-\d{4}$")).select(
    F.substring("data_venda", 1, 2).cast("int").alias("p1"),
    F.substring("data_venda", 4, 2).cast("int").alias("p2"),
)
display(x.select(
    F.when(F.col("p1") > 12, "dd-MM (certo)")
     .when(F.col("p2") > 12, "MM-dd (certo)")
     .otherwise("ambíguo").alias("classe")
).groupBy("classe").count())

In [ ]:
display(df.select(
    F.sum(F.when(F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"), 1).otherwise(0)).alias("cpf_formato_ok"),
    F.sum(F.when(~F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$") & F.col("cliente_cpf").isNotNull(), 1).otherwise(0)).alias("cpf_formato_fora"),
    F.sum(F.when(F.col("cliente_telefone").rlike(r"^\(\d{2}\) \d{5}-\d{4}$"), 1).otherwise(0)).alias("tel_formato_ok"),
    F.sum(F.when(F.col("cliente_email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), 1).otherwise(0)).alias("email_formato_ok"),
))

# Amostra de CPFs fora do padrão
display(df.filter(~F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$")).select("cliente_cpf").distinct().limit(15))

In [ ]:
for c in ["valor_frete", "qtd_vendida", "valor_unitario", "avaliacao_produto", "desconto_pct"]:
    n = F.expr(f"try_cast({c} as double)")
    print(f"\n=== {c} ===")
    display(df.select(
        F.sum(F.when(n.isNull() & F.col(c).isNotNull(), 1).otherwise(0)).alias("nao_numerico"),
        F.min(n).alias("min"),
        F.expr(f"percentile_approx(try_cast({c} as double), 0.5)").alias("mediana"),
        F.max(n).alias("max"),
        F.sum(F.when(n < 0, 1).otherwise(0)).alias("negativos"),
    ))    

In [ ]:
def parse_data(c):
    return F.coalesce(
        F.expr(f"try_to_date({c}, 'yyyy-MM-dd')"),
        F.expr(f"try_to_date({c}, 'dd/MM/yyyy')"),
        F.expr(f"try_to_date({c}, 'MM-dd-yyyy')"),
    )

base = (df
    .withColumn("dv", parse_data("data_venda"))
    .withColumn("dr", parse_data("data_retirada"))
    .withColumn("qtd", F.expr("try_cast(qtd_vendida as int)"))
    .withColumn("pct", F.expr("try_cast(desconto_pct as double)"))
    .withColumn("st_envio", F.upper(F.trim("status_envio")))
    .withColumn("st_venda", F.upper(F.trim("status_venda"))))

display(base.select(
    F.sum(F.when(F.col("dv").isNull(), 1).otherwise(0)).alias("data_venda_nao_interpretada"),
    F.sum(F.when(F.col("dr").isNull(), 1).otherwise(0)).alias("data_retirada_nao_interpretada"),
    F.sum(F.when(F.col("dr") < F.col("dv"), 1).otherwise(0)).alias("retirada_antes_da_venda"),
    F.sum(F.when((F.col("status_desconto") == "Não") & (F.col("pct") > 0), 1).otherwise(0)).alias("sem_desconto_mas_com_pct"),
    F.sum(F.when((F.col("status_desconto") == "Sim") & (F.col("pct") == 0), 1).otherwise(0)).alias("com_desconto_mas_pct_zero"),
    F.sum(F.when((F.col("st_venda") == "CANCELADA") & F.col("st_envio").isin("ENVIADO", "ENTREGUE", "EM TRÂNSITO"), 1).otherwise(0)).alias("cancelada_mas_enviada"),
    F.sum(F.when(F.col("qtd") <= 0, 1).otherwise(0)).alias("qtd_zero_ou_negativa"),
    F.min("dv").alias("data_venda_min"), F.max("dv").alias("data_venda_max"),
))

display(df.groupBy("produto", "tipo_produto").count().orderBy("produto"))

In [ ]:
display(df.filter(padrao_data("data_retirada") == "outro")
          .select("data_retirada").distinct().limit(15))

# Confirmar que o formato com barra em data_venda é mesmo dd/MM
s = df.filter(F.col("data_venda").rlike(r"^\d{2}/\d{2}/\d{4}$")).select(
    F.substring("data_venda", 1, 2).cast("int").alias("p1"),
    F.substring("data_venda", 4, 2).cast("int").alias("p2"))
display(s.select(
    F.when(F.col("p1") > 12, "dd/MM (certo)")
     .when(F.col("p2") > 12, "MM/dd (certo)")
     .otherwise("ambíguo").alias("classe")).groupBy("classe").count())

In [ ]:
for c in ["estado", "status_envio", "cidade"]:
    print(f"\n=== {c} ===")
    print(df.groupBy(c).count().orderBy(c).toPandas().to_string())

In [ ]:
display(df.filter(F.col("cliente_cpf").isNotNull() &
                  ~F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"))
          .select("cliente_cpf").distinct().limit(15))

cpfs = (df.filter(F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"))
          .select("cliente_cpf").distinct().toPandas()["cliente_cpf"])

def cpf_valido(cpf):
    d = [int(x) for x in cpf if x.isdigit()]
    if len(set(d)) == 1:
        return False
    for i in (9, 10):
        s = sum(d[j] * (i + 1 - j) for j in range(i))
        if (s * 10 % 11) % 10 != d[i]:
            return False
    return True

print(cpfs.map(cpf_valido).value_counts())

In [ ]:
display(df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in colunas_negocio]))

chave = ["cliente", "produto", "data_venda"]
display(df.groupBy(chave).count().filter("count > 1").orderBy(F.desc("count")).limit(10))

In [ ]:
r = df.filter(F.col("data_retirada").rlike(r"^\d{2}-\d{2}-\d{4}$")).select(
    F.substring("data_retirada", 1, 2).cast("int").alias("p1"),
    F.substring("data_retirada", 4, 2).cast("int").alias("p2"))

display(r.select(
    F.when(F.col("p1") > 12, "dd-MM (certo)")
     .when(F.col("p2") > 12, "MM-dd (certo)")
     .otherwise("ambíguo").alias("classe")).groupBy("classe").count())

In [ ]:
cid = F.lower(F.trim("cidade")).alias("cidade_norm")

# 1) Em linhas com UF limpa: cada cidade tem uma UF só?
limpo = df.filter(F.col("estado").rlike(r"^[A-Z]{2}$"))
print(limpo.groupBy(cid).agg(F.collect_set("estado").alias("ufs"), F.count("*").alias("n"))
           .orderBy("cidade_norm").toPandas().to_string())

# 2) Com quais cidades aparecem os estados "estranhos"?
estranho = df.filter(~F.col("estado").rlike(r"^[A-Za-z]{2}$"))
print(estranho.groupBy("estado", cid).count().orderBy("estado", F.desc("count")).toPandas().to_string())

In [ ]:
display(df.filter(~F.col("cliente_email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))
          .select("cliente", "cliente_email").distinct().limit(15))

display(df.filter(F.col("cliente_cpf").isNotNull() &
                  ~F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"))
          .select(F.when(F.col("cliente_cpf").rlike(r"^\d{11}X$"), "11 dígitos + X")
                   .when(F.col("cliente_cpf") == "123", "valor 123")
                   .otherwise("outro").alias("tipo"))
          .groupBy("tipo").count())